# Sakura → Zakuro async-training benchmark

Walks through the three stages of Sakura's async-eval pipeline on a BERT fine-tune:

1. **Serial baseline** — the `transformers.Trainer` default: one epoch trains, then the same process runs the full eval loop before the next epoch starts.
2. **Sakura naïve async** — eval goes to a Zakuro worker via `@zk.fn`; training no longer blocks, but the callback still has to cloudpickle a full state-dict each epoch.
3. **Sakura + `zk.AdaptiveCompute`** — the Adam-style allocator tracks per-worker latency EMA, exposes a backpressure signal, and lets the callback *skip* an epoch's eval when the slow side can't keep up. Training stays unblocked.

The notebook uses Zakuro's standalone fallback (in-process execution) by default so it runs on a laptop with no worker setup. Set `WORKER_URI` to point at a real QUIC worker for cross-machine runs.

In [ ]:
import os
import time

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

import zakuro as zk
from sakura.huggingface import SakuraHFCallback

print("torch:", torch.__version__)
print("zakuro:", zk.__version__)
print("has AdaptiveCompute:", hasattr(zk, "AdaptiveCompute"))

## Configuration

Knobs you can tweak. The defaults are intentionally small so the notebook runs in a couple of minutes end-to-end.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
TRAIN_SIZE = 600
VAL_SIZE = 1200
MAX_LENGTH = 64
EPOCHS = 4
BATCH_SIZE = 32

# Leave None to use Zakuro's standalone fallback. Set to e.g.
# "quic://192.168.0.220:4445" to target a real worker.
WORKER_URI = os.environ.get("SAKURA_VAL_WORKER", None)

## Data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
raw = load_dataset("glue", "sst2")

def tokenize(batch):
    return tokenizer(
        batch["sentence"], padding="max_length", truncation=True, max_length=MAX_LENGTH
    )

train = raw["train"].shuffle(seed=42).select(range(TRAIN_SIZE)).map(tokenize, batched=True)
val = raw["validation"].shuffle(seed=42).select(range(min(VAL_SIZE, len(raw["validation"])))).map(
    tokenize, batched=True
)
cols = ["input_ids", "attention_mask", "label"]
train.set_format("torch", columns=cols)
val.set_format("torch", columns=cols)
print(f"train: {len(train)}  val: {len(val)}")

## Helpers

A single `model_factory` + `eval_fn` pair used by both async modes. Imports live inside `eval_fn` so it cloudpickles cleanly for cross-Python-version workers.

In [ ]:
_config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)

def model_factory():
    # Architecture only — Sakura streams the weights over each epoch.
    return AutoModelForSequenceClassification.from_config(_config)

def eval_fn(model, payload):
    import torch as _torch

    data, bs = payload
    if _torch.cuda.is_available():
        device = _torch.device("cuda")
    elif getattr(_torch.backends, "mps", None) and _torch.backends.mps.is_available():
        device = _torch.device("mps")
    else:
        device = _torch.device("cpu")
    model.to(device)

    loss_sum = 0.0
    correct = 0
    total = 0
    with _torch.no_grad():
        for start in range(0, len(data["input_ids"]), bs):
            batch = {k: data[k][start:start+bs].to(device) for k in ("input_ids", "attention_mask")}
            labels = data["label"][start:start+bs].to(device)
            out = model(**batch, labels=labels)
            loss_sum += float(out.loss.item()) * labels.size(0)
            correct += int((out.logits.argmax(-1) == labels).sum().item())
            total += int(labels.size(0))
    return {"val_loss": loss_sum / max(total, 1), "val_acc": correct / max(total, 1)}

# Materialise the val tensors so they pickle cleanly (no Arrow backing).
val_payload = (
    {
        "input_ids": torch.stack([val[i]["input_ids"] for i in range(len(val))]).clone(),
        "attention_mask": torch.stack([val[i]["attention_mask"] for i in range(len(val))]).clone(),
        "label": torch.stack([val[i]["label"] for i in range(len(val))]).clone(),
    },
    BATCH_SIZE,
)
print("helpers ready")

In [ ]:
def _args(outdir, eval_strategy):
    return TrainingArguments(
        output_dir=outdir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy=eval_strategy,
        save_strategy="no",
        logging_strategy="no",
        report_to=[],
        disable_tqdm=True,
        seed=42,
        dataloader_num_workers=0,
        fp16=torch.cuda.is_available(),
    )

## 1. Serial baseline

`transformers.Trainer` with `eval_strategy="epoch"`. Evaluation happens inline — training waits.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer = Trainer(
    model=model,
    args=_args("/tmp/sakura-nb/serial", "epoch"),
    train_dataset=train,
    eval_dataset=val,
)

t0 = time.perf_counter()
trainer.train()
serial_wall = time.perf_counter() - t0
serial_final_loss = trainer.evaluate()["eval_loss"]
print(f"serial: {serial_wall:.2f}s  final val_loss={serial_final_loss:.4f}")

## 2. Sakura naïve async

`SakuraHFCallback` attached; `eval_strategy="no"` so `Trainer` doesn't try to eval. The callback ships each epoch's state-dict to a Zakuro worker (or standalone fallback).

No backpressure yet — the callback dispatches unconditionally. Training runs at the mercy of whichever side is slower.

In [ ]:
if WORKER_URI:
    compute = zk.Compute(uri=WORKER_URI, verify=False)
    print(f"async eval → {compute.uri}")
else:
    compute = zk.Compute()  # standalone fallback
    print("async eval → zakuro standalone (in-process)")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
cb_naive = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=compute,
    drain="lazy",
    cache_key=f"nb-{MODEL_NAME}",
    fp16_state_dict=True,
    on_backpressure="queue",    # always dispatch, even under load
    verbose=False,
)
trainer = Trainer(
    model=model,
    args=_args("/tmp/sakura-nb/naive", "no"),
    train_dataset=train,
    callbacks=[cb_naive],
)

t0 = time.perf_counter()
trainer.train()
naive_wall = time.perf_counter() - t0
print(f"naïve async: {naive_wall:.2f}s")
for row in cb_naive.history:
    if row.get("skipped"):
        print(f"  epoch {int(row['epoch'])}: SKIPPED")
    else:
        print(f"  epoch {int(row['epoch'])}: loss={row['val_loss']:.4f} acc={row['val_acc']:.4f} took={row['elapsed_secs']:.2f}s")

## 3. Sakura + `zk.AdaptiveCompute`

Same setup, but `val_compute` is wrapped in `AdaptiveCompute`. The allocator tracks per-worker latency EMA (Adam-style), exposes `is_backpressured()`, and the callback's `on_backpressure="skip"` lets training continue when the slow side can't keep up.

With a single worker, adaptive's main value is the backpressure signal. With multiple workers it also routes by expected time-to-serve.

In [ ]:
adaptive = zk.AdaptiveCompute(
    workers=[compute],
    beta1=0.7,         # responsive to recent observations
    beta2=0.99,
    backpressure_threshold=3.0,  # seconds
    initial_latency=0.5,
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
cb_adaptive = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=adaptive,
    drain="lazy",
    cache_key=f"nb-{MODEL_NAME}",
    fp16_state_dict=True,
    on_backpressure="skip",     # drop this epoch's eval when overloaded
    verbose=False,
)
trainer = Trainer(
    model=model,
    args=_args("/tmp/sakura-nb/adaptive", "no"),
    train_dataset=train,
    callbacks=[cb_adaptive],
)

t0 = time.perf_counter()
trainer.train()
adaptive_wall = time.perf_counter() - t0
print(f"adaptive: {adaptive_wall:.2f}s")
evals = [r for r in cb_adaptive.history if not r.get("skipped")]
skips = [r for r in cb_adaptive.history if r.get("skipped")]
for row in evals:
    print(f"  epoch {int(row['epoch'])}: loss={row['val_loss']:.4f} acc={row['val_acc']:.4f} took={row['elapsed_secs']:.2f}s")
print(f"  skipped by backpressure: {len(skips)} epoch(s)")
print("\nallocator stats:")
for i, s in enumerate(adaptive.stats()):
    print(f"  worker {i}: step={s['step']} queue={s['queue']} ema={s['latency_ema']:.3f}s var={s['latency_var']:.3f}")

## 4. Comparison

Wall-clock summary. Rows should show: serial competes well on the fast side; async naïve can lose for small per-epoch eval times because the dispatch overhead exceeds savings; adaptive approaches serial from below by skipping evals the mesh can't absorb.

In [ ]:
rows = [
    ("serial", serial_wall),
    ("naive async", naive_wall),
    ("adaptive + skip", adaptive_wall),
]
width = 20
print(f"{'mode':<{width}}{'wall (s)':>10}{'Δ vs serial':>14}")
print("-" * (width + 24))
for name, wall in rows:
    delta = wall - serial_wall
    pct = 100 * delta / serial_wall
    arrow = "" if name == "serial" else f"{'+' if delta > 0 else ''}{delta:.2f}s ({pct:+.1f}%)"
    print(f"{name:<{width}}{wall:>10.2f}{arrow:>14}")

In [ ]:
# Optional: plot a simple bar chart if matplotlib is around.
try:
    import matplotlib.pyplot as plt
    names = [r[0] for r in rows]
    vals = [r[1] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(names, vals, color=["#8bb174", "#d4a373", "#ae5577"])
    ax.set_ylabel("wall-clock seconds")
    ax.set_title(f"{MODEL_NAME} • {EPOCHS} epochs • val size={VAL_SIZE}")
    for name, v in zip(names, vals):
        ax.text(name, v, f"{v:.1f}s", ha="center", va="bottom")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping plot.")

## Notes

- On a single machine (standalone fallback) the async path **can lose** to serial if eval is much cheaper than training: the cloudpickle + thread-pool overhead eats the potential savings. This is honest; the framework can't overcome that physics.
- Async **wins** when:
  - Training time per epoch is comparable to or greater than eval time.
  - Eval contains CPU-bound work (seqeval, BLEU, BERTScore) where Mac ≈ x399.
  - Multiple eval workers are available and the allocator can fan out.
  - You care about metric sampling rate more than per-epoch granularity — adaptive's skip policy trades some metrics for faster wall-clock.
- See [`zakuro/PLAN.md`](https://github.com/zakuro-ai/zakuro/blob/master/PLAN.md) for the roadmap toward grid-aware scheduling and truly-decoupled checkpoint handling, which should push async well past serial on typical workloads.